## Prompt Engineering

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [4]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

### Zero-shot vs One-shot vs Few-shot

In [2]:
zero_shot = """
Classify the sentiment of the following review:

"The delivery was delayed by two weeks."
"""

one_shot = """
Example:

Review:
"The product is excellent."

Sentiment:
positive

Now classify:

"The delivery was delayed by two weeks."
"""

few_shot = """
Review:
"The product is excellent."
Sentiment:
positive

Review:
"The package arrived broken."
Sentiment:
negative

Review:
"The order was delivered."
Sentiment:
neutral

Now classify:

"The delivery was delayed by two weeks."
"""

In [5]:
import pandas as pd

prompts = {
    "Zero-shot": zero_shot,
    "One-shot": one_shot,
    "Few-shot": few_shot
}

results = []

for prompt_name, prompt in prompts.items():

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    results.append({
        "Prompt Type": prompt_name,
        "Response": response.output_text,
        "Input Tokens": response.usage.input_tokens,
        "Output Tokens": response.usage.output_tokens,
        "Total Tokens": response.usage.total_tokens
    })

df = pd.DataFrame(results)

df

,Prompt Type,Response,Input Tokens,Output Tokens,Total Tokens
0,Zero-shot,"The sentiment of the review ""The delivery was ...",25,18,43
1,One-shot,Sentiment: \nnegative,33,6,39
2,Few-shot,Sentiment: \nnegative,55,6,61


### Observation

One-shot and few-shot prompting improved adherence to the desired output format.

The zero-shot prompt produced a longer natural-language explanation, while prompts with examples generated concise and consistent outputs.

Examples help the model understand both the task and the expected response structure.

In [6]:
import pandas as pd

reviews = [
    ("The delivery was delayed by two weeks and customer support never replied.", "negative"),
    ("The product quality is excellent, but delivery took three weeks.", "neutral"),
    ("Customer support was helpful, although the issue took several days to resolve.", "neutral"),
    ("The order arrived on time, but the packaging was damaged.", "neutral"),
    ("I received a refund immediately after reporting the problem.", "positive"),
    ("The product works exactly as described.", "positive"),
    ("The package arrived broken and the replacement never came.", "negative"),
    ("Everything was fine.", "positive"),
    ("The item arrived yesterday.", "neutral"),
    ("I am extremely disappointed with the service.", "negative")
]

results = []

for review, expected in reviews:

    zero_shot = f"""
Classify the sentiment of the following review.

Return only one word:
positive, negative, or neutral.

Review:
{review}
"""

    one_shot = f"""
Example:

Review:
"The product is excellent and arrived quickly."

Sentiment:
positive

Now classify.

Return only one word:
positive, negative, or neutral.

Review:
{review}
"""

    few_shot = f"""
Review:
"The product is excellent and arrived quickly."
Sentiment:
positive

Review:
"The package arrived damaged."
Sentiment:
negative

Review:
"The order was delivered yesterday."
Sentiment:
neutral

Now classify.

Return only one word:
positive, negative, or neutral.

Review:
{review}
"""

    row = {
        "Review": review,
        "Expected": expected
    }

    prompts = {
        "Zero-shot": zero_shot,
        "One-shot": one_shot,
        "Few-shot": few_shot
    }

    for prompt_name, prompt in prompts.items():

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt
        )

        prediction = response.output_text.strip().lower()

        row[prompt_name] = prediction

    results.append(row)

df = pd.DataFrame(results)

df

,Review,Expected,Zero-shot,One-shot,Few-shot
0,The delivery was delayed by two weeks and cust...,negative,negative,negative,negative
1,"The product quality is excellent, but delivery...",neutral,neutral,neutral,neutral
2,"Customer support was helpful, although the iss...",neutral,neutral,neutral,neutral
3,"The order arrived on time, but the packaging w...",neutral,neutral,neutral,neutral
4,I received a refund immediately after reportin...,positive,positive,positive,positive
5,The product works exactly as described.,positive,positive,positive,positive
6,The package arrived broken and the replacement...,negative,negative,negative,negative
7,Everything was fine.,positive,neutral,neutral,positive
8,The item arrived yesterday.,neutral,neutral,neutral,neutral
9,I am extremely disappointed with the service.,negative,negative,negative,negative


In [7]:
accuracy_results = []

for prompt_type in ["Zero-shot", "One-shot", "Few-shot"]:

    accuracy = (
        df[prompt_type] == df["Expected"]
    ).mean()

    accuracy_results.append({
        "Prompt Type": prompt_type,
        "Accuracy": round(accuracy, 2)
    })

accuracy_df = pd.DataFrame(accuracy_results)

accuracy_df

,Prompt Type,Accuracy
0,Zero-shot,0.9
1,One-shot,0.9
2,Few-shot,1.0


### Observation

Few-shot prompting achieved the highest accuracy on the evaluation dataset.

Providing multiple examples helped the model better understand the classification task and expected output format.

In this experiment, few-shot prompting improved performance from 90% to 100% accuracy.